# Aşama 1 — LR Pilotu Entegrasyon Testi

Bu notebook bilimsel LR pilotu sonucu üretmez. Gerçek `calistir_lr_pilotu.py` akışını Drive veri yolu, kalıcı çıktı, checkpoint/resume ve yeni metriklerle **yalnız ilk koşu / 3 epoch** çalıştırarak entegrasyonu doğrular.


In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)


## Güncel public GitHub sürümünü al


In [ ]:
import io, os, shutil, tarfile, requests
from pathlib import Path

OWNER='erdemci07'
REPO='derin-ogrenme-egitim-dinamikleri'
BRANCH='main'
repo_dir=Path('/content/derin-ogrenme-egitim-dinamikleri')
extract_root=Path('/content/_tez_repo_extract')
shutil.rmtree(repo_dir, ignore_errors=True)
shutil.rmtree(extract_root, ignore_errors=True)
extract_root.mkdir(parents=True, exist_ok=True)
r=requests.get(f'https://api.github.com/repos/{OWNER}/{REPO}/commits/{BRANCH}', timeout=60)
r.raise_for_status()
commit_sha=r.json()['sha']
a=requests.get(f'https://api.github.com/repos/{OWNER}/{REPO}/tarball/{commit_sha}', timeout=120)
a.raise_for_status()
with tarfile.open(fileobj=io.BytesIO(a.content), mode='r:gz') as tar:
    tar.extractall(extract_root)
kokler=[p for p in extract_root.iterdir() if p.is_dir()]
if len(kokler)!=1:
    raise RuntimeError('GitHub arşiv yapısı beklenenden farklı.')
shutil.move(str(kokler[0]), str(repo_dir))
os.chdir(repo_dir)
os.environ['TEZ_GIT_COMMIT']=commit_sha
print('Repo hazır:', commit_sha)


## Google Drive yolları


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_KOK=Path('/content/drive/MyDrive')
DRIVE_VERI=DRIVE_KOK/'tez_datasets'
DRIVE_PROJE=DRIVE_KOK/'tez_deney_sonuclari'
DRIVE_AYAR=DRIVE_PROJE/'ortak_ayarlar'
DRIVE_CIKTI=DRIVE_PROJE/'lr_pilotu_entegrasyon_testi'
if not DRIVE_VERI.is_dir():
    raise FileNotFoundError(f'Veri klasörü bulunamadı: {DRIVE_VERI}')
DRIVE_AYAR.mkdir(parents=True, exist_ok=True)
DRIVE_CIKTI.mkdir(parents=True, exist_ok=True)
SPLIT_DOSYASI=DRIVE_AYAR/'cifar10_split.json'
STATS_DOSYASI=DRIVE_AYAR/'cifar10_stats.json'
os.environ['TEZ_VERI_KLASORU']=str(DRIVE_VERI)
os.environ['TEZ_SPLIT_DOSYASI']=str(SPLIT_DOSYASI)
os.environ['TEZ_STATS_DOSYASI']=str(STATS_DOSYASI)
os.environ['TEZ_CIKTI_KLASORU']=str(DRIVE_CIKTI)
print('Veri:',DRIVE_VERI)
print('Çıktı:',DRIVE_CIKTI)


## Bağımlılık ve CUDA kontrolü


In [ ]:
subprocess.run(['python','-m','pip','install','-q','-r','requirements_colab.txt'], check=True)
import torch
print('PyTorch:',torch.__version__)
print('CUDA:',torch.cuda.is_available())
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK')
if not torch.cuda.is_available():
    raise RuntimeError('T4/CUDA aktif değil.')


## Gerçek LR pilotu motoruyla ilk koşuyu 3 epoch çalıştır

Bu test yalnızca ilk planlanan LR koşusunu çalıştırır. `--epoch 3` yalnız bu teknik test içindir; gerçek pilotta YAML'daki 30 epoch kullanılacaktır.


In [ ]:
import importlib
import calistir_lr_pilotu as lr
importlib.reload(lr)
lr.ana(['--calistir','--maksimum-run','1','--epoch','3'])


## Çıktıyı doğrula


In [ ]:
import csv, json, math
run_klasorleri=[p for p in DRIVE_CIKTI.iterdir() if p.is_dir()]
if len(run_klasorleri)!=1:
    raise RuntimeError(f'Beklenen 1 run klasörü yerine {len(run_klasorleri)} bulundu.')
run_dir=run_klasorleri[0]
tamam=run_dir/'tamamlandi.json'
csv_yolu=run_dir/'epoch_sonuclari.csv'
if not tamam.exists() or not csv_yolu.exists():
    raise RuntimeError('Tamamlandı JSON veya epoch CSV oluşmadı.')
ozet=json.loads(tamam.read_text(encoding='utf-8'))
with csv_yolu.open(encoding='utf-8', newline='') as f:
    satirlar=list(csv.DictReader(f))
kontroller={
 'epoch_3': len(satirlar)==3 and int(ozet['epoch'])==3,
 'macro_f1_var': all('macro_f1' in x and math.isfinite(float(x['macro_f1'])) for x in satirlar),
 'aulc_var': math.isfinite(float(ozet['aulc_normalized'])),
 'epoch95_var': 1 <= int(ozet['epoch_to_95_own_best']) <= 3,
 'gpu_t4': 'T4' in str(ozet.get('gpu','')).upper(),
 'git_commit_var': ozet.get('git_commit') not in (None,'','unknown'),
 'checkpoint_temizlendi': not (run_dir/'checkpoint.pt').exists(),
}
print('Run:',run_dir.name)
print('Kontroller:',kontroller)
print('Özet:',ozet)
if not all(kontroller.values()):
    raise RuntimeError('LR pilotu entegrasyon testi başarısız.')
print('\nLR PİLOTU ENTEGRASYON TESTİ: GEÇTİ')
